# Data Engineering Foundations

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/T31SHA/data-ml-handbook/blob/main/notebooks/16_data_engineering.ipynb)

Runnable examples from **Data/ML Handbook** — section `16_data_engineering`. Read the explanations in the [docs](https://t31sha.github.io/data-ml-handbook/handbook/16_data_engineering/).

In [ ]:
# Setup: works in Colab or locally from the repo root
import os, subprocess, sys
if not os.path.exists("code") and not os.path.exists("../code"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/T31SHA/data-ml-handbook.git"], check=True)
    os.chdir("data-ml-handbook")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if os.path.exists("../code"): os.chdir("..")
os.chdir("code"); os.makedirs("data", exist_ok=True)
for prep in ["proj01_make_csv.py", "s02_load.py"]:          # shared datasets used by many snippets
    if not os.path.exists("data/housing.parquet"): subprocess.run([sys.executable, prep], check=True)
sys.path.insert(0, "../figures/src")

### `code/s13_idempotent.py`

In [ ]:
import pandas as pd, pyarrow as pa, pyarrow.parquet as pq, shutil, os, glob
root = "data/lake/orders"; shutil.rmtree(root, ignore_errors=True)
def load_partition(ds: str, df: pd.DataFrame):
    """Idempotent: overwrite exactly one date partition; re-runs/backfills produce the same result."""
    pq.write_to_dataset(pa.Table.from_pandas(df.assign(dt=ds)), root, partition_cols=["dt"],
                        existing_data_behavior="delete_matching")
day = pd.DataFrame({"order_id": [1, 2, 3], "amount": [10.0, 20.0, 5.0]})
load_partition("2026-10-23", day); load_partition("2026-10-23", day); load_partition("2026-10-24", day.head(1))
t = pq.read_table(root).to_pandas()
print(t.groupby("dt", observed=True).size().to_string(), "\nfiles:", sorted(os.path.relpath(p, root) for p in glob.glob(root + "/*/*")))

### `code/s13_quality.py`

In [ ]:
import pandas as pd, numpy as np
import pandera.pandas as pa
schema = pa.DataFrameSchema({
    "order_id": pa.Column(int, unique=True),
    "amount": pa.Column(float, [pa.Check.ge(0), pa.Check.le(1e6)]),
    "status": pa.Column(str, pa.Check.isin(["new", "paid", "refund"])),
    "email": pa.Column(str, pa.Check.str_matches(r"^[^@\s]+@[^@\s]+\.\w+$"), nullable=True)}, strict=True)
bad = pd.DataFrame({"order_id": [1, 2, 2], "amount": [10.0, -5.0, 30.0], "status": ["paid", "PAID", "new"],
                    "email": ["a@x.com", "nope", None]})
try: schema.validate(bad, lazy=True)
except pa.errors.SchemaErrors as e:
    print(e.failure_cases[["column", "check", "failure_case"]].to_string(index=False))